# Synthetic contact-form messages for Local AI Anti-Spam

Generates realistic genuine messages **and** spam for training the spam filter (PRD D6, review
notes R13/R14). Built from `training/laas/synth/` by `python -m laas.synth.notebook`; don't edit
this notebook by hand.

**Run it on Kaggle (recommended)**
1. *Settings* (right panel): **Accelerator: GPU T4 x2**, **Internet: On** (needs a phone-verified
   account).
2. Leave `PILOT = True` for the first run (about 1,000 messages, roughly 20-40 minutes).
   Leave `USE_VLLM = False` unless you want to try the faster engine (it runs isolated, so it
   can't break anything; if it fails the run continues with transformers).
3. **Save Version, then Save & Run All (Commit)**, or run the cells one by one.
4. Every cell prints a ▶ line when it starts and a ✅ (or ❌) line when it ends. Step 4 prints a
   status line every 30 seconds while it generates.
5. When it finishes: tab **Output**, download `synthetic.zip`, and unzip it into
   `training/data/synthetic/` in the repo.

**On Colab:** Runtime, Change runtime type, **T4 GPU**, then Run all. The output is under
`/content/synthetic`; download `synthetic.zip` from the file browser.

Only invented messages are generated here. Never upload real beta-site messages to Kaggle or Colab.


## 1. Settings

In [ ]:
print("▶ Step 1/5: settings")
import os

PILOT = True  # True: ~1,000 messages to check quality. False: the full run (~40,000 messages).
N_REQUESTS = 100 if PILOT else 4000  # per generator; each request asks for 5 messages

# False (default): use Kaggle's own PyTorch + transformers. Nothing big is installed, so nothing breaks.
# True: also try vLLM (much faster) in a separate virtual environment, falling back to transformers.
# Never pip-install vLLM into the main environment: it replaces Kaggle's PyTorch and breaks transformers.
USE_VLLM = False

GENERATORS = [
    # Apache-2.0 (checked 18 Oct 2026). AWQ fits one T4; the 4-bit fallback is for transformers.
    dict(generator="qwen2.5-7b", model="Qwen/Qwen2.5-7B-Instruct-AWQ", quantization="awq",
         fallback_model="Qwen/Qwen2.5-7B-Instruct", fallback_4bit=True),
    # MIT. A different model family, so the data isn't one model's writing style.
    dict(generator="phi-4-mini", model="microsoft/Phi-4-mini-instruct", quantization="",
         fallback_model="microsoft/Phi-4-mini-instruct", fallback_4bit=False),
]

OUT_DIR = "/kaggle/working/synthetic" if os.path.isdir("/kaggle/working") else "/content/synthetic"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"   mode: {'PILOT' if PILOT else 'FULL RUN'} · {N_REQUESTS} requests x 5 messages per model")
print(f"   models: {', '.join(g['generator'] for g in GENERATORS)}")
print(f"   engine: {'vLLM (isolated) with transformers fallback' if USE_VLLM else 'transformers'}")
print(f"   output folder: {OUT_DIR}")
print("✅ Step 1 done")


## 2. GPU check and install

In [ ]:
print("▶ Step 2/5: checking the GPU environment")
import subprocess, sys

def run(cmd):
    return subprocess.run(cmd, capture_output=True, text=True)

def last_line(result):
    return (result.stderr.strip().splitlines() or result.stdout.strip().splitlines() or ["?"])[-1]

gpu = run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
print("   GPUs:", gpu.stdout.strip().replace("\n", " | ") or "NONE FOUND: turn on a GPU accelerator")

# bitsandbytes (4-bit loading) is the only addition, and --no-deps stops it touching Kaggle's PyTorch.
if run([sys.executable, "-c", "import bitsandbytes"]).returncode != 0:
    print("   installing bitsandbytes (without touching PyTorch)...")
    run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "bitsandbytes"])

# Import the real model modules now, so a broken environment fails here in seconds, not in step 4.
# Importing the modules directly (not via `from transformers import ...`) shows the real error
# instead of transformers' generic "Could not import module" wrapper.
PROBE = '''
import importlib, sys, traceback
try:
    import torch, transformers
    print(f"PyTorch {torch.__version__} (CUDA {torch.version.cuda}) · transformers {transformers.__version__}"
          f" · GPU available: {torch.cuda.is_available()}", flush=True)
    try:
        import torchvision
        print(f"torchvision {torchvision.__version__}", flush=True)
    except Exception as exc:
        print(f"torchvision: not usable ({type(exc).__name__}: {exc})", flush=True)
    for name in ("accelerate", "bitsandbytes",
                 "transformers.models.qwen2.modeling_qwen2", "transformers.models.phi3.modeling_phi3"):
        importlib.import_module(name)
        print(f"ok: {name}", flush=True)
except Exception:
    traceback.print_exc()
    sys.exit(1)
'''
probe = run([sys.executable, "-c", PROBE])
for line in probe.stdout.strip().splitlines():
    print("  ", line)
if probe.returncode != 0:
    print("❌ Step 2 failed: PyTorch/transformers can't load the models in this session.")
    print("   Real error (last lines):")
    print("\n".join("     " + l for l in probe.stderr.strip().splitlines()[-12:]))
    print("   Fix 1: fully reset the session: click the power icon at the top and wait for")
    print("          'Session off' (or Run > Factory reset), then Run All. Restarting the kernel")
    print("          alone does NOT undo packages installed earlier.")
    print("   Fix 2: if it still fails after a full reset, send these lines over.")
    raise SystemExit("environment check failed")

VLLM_PYTHON = None
if USE_VLLM:
    print("   USE_VLLM: installing vLLM into its own virtual environment (5-10 minutes)...")
    venv = "/tmp/vllm-env"
    run([sys.executable, "-m", "venv", venv])
    pip = run([f"{venv}/bin/python", "-m", "pip", "install", "-q", "vllm"])
    check = run([f"{venv}/bin/python", "-c", "import vllm; print(vllm.__version__)"])
    if check.returncode == 0:
        VLLM_PYTHON = f"{venv}/bin/python"
        print(f"   vLLM {check.stdout.strip()} ready (isolated; the main environment is untouched)")
    else:
        print("   ⚠ vLLM didn't install or import; using transformers. Reason:", last_line(check))
print("✅ Step 2 done")


## 3. Generation code (copied from `training/laas/synth/`)

In [ ]:
%%writefile spec.py
"""Synthetic contact-form data: the generation grid and prompts (W1-10, PRD D6).

Single source of truth for what the generator writes. The Kaggle notebook embeds this file verbatim
(``python -m laas.synth.notebook``), so keep it standard-library only.

Design driven by review notes R13/R14: the public data taught the model "casual = spam", so genuine
messages here deliberately cover email-style, chat-style, short, rambling, non-native and angry
writing, and spam covers short comment spam and human-sounding SEO/web-design pitches.

A *template* is (prompt version, channel, intent, style). Splits are made by template (PRD D4), so
the test side never contains an intent/style combination the model trained on.
"""

from __future__ import annotations

import hashlib
import json
import random
import re
from dataclasses import asdict, dataclass

# v2 (18 Oct 2026, after the pilot review): genuine writers sometimes include links and emails (v1
# had none, which would teach "link = spam"); spam briefs say who is selling to whom (v1's loan
# offers came out as customers asking for loans); stricter artefact filter.
PROMPT_VERSION = 2
MESSAGES_PER_REQUEST = 5

# (business, what a customer would contact it about). At least 20 types (PRD D6).
BUSINESSES = [
    ("plumber", "leaks, boilers, bathroom fitting, emergency call-outs"),
    ("dental practice", "check-ups, emergency appointments, whitening, NHS or private fees"),
    ("family restaurant", "table bookings, private parties, menus, allergies"),
    ("law firm", "wills, conveyancing, employment disputes, initial consultations"),
    ("accountant", "tax returns, bookkeeping, VAT, small-business accounts"),
    ("hair salon", "appointments, colouring, bridal hair, prices"),
    ("gym", "memberships, personal training, cancellation, classes"),
    ("wedding photographer", "availability, packages, travel, albums"),
    ("estate agent", "valuations, viewings, lettings, selling a house"),
    ("roofing contractor", "leaks, new roofs, guttering, quotes"),
    ("veterinary clinic", "appointments, vaccinations, emergencies, pet insurance"),
    ("bakery", "custom cakes, wholesale orders, opening hours, allergens"),
    ("driving school", "lessons, intensive courses, test dates, prices"),
    ("IT support company", "broken laptops, office networks, backups, contracts"),
    ("yoga studio", "class times, beginners, private sessions, memberships"),
    ("landscaping company", "garden design, lawn care, fencing, quotes"),
    ("car repair garage", "MOT or inspection, servicing, warning lights, quotes"),
    ("private tutor", "maths and English tutoring, exam prep, online lessons"),
    ("cleaning company", "end-of-tenancy cleans, office cleaning, regular cleaners"),
    ("clothing boutique", "returns, sizes, stock, alterations"),
    ("bed and breakfast", "room availability, group bookings, parking, pets"),
    ("florist", "wedding flowers, funeral wreaths, same-day delivery"),
    ("physiotherapist", "back pain, sports injuries, appointments, insurance"),
    ("nursery and daycare", "places, fees, settling-in, opening hours"),
    ("electrician", "rewiring, sockets, EV chargers, safety certificates"),
    ("removals company", "house moves, quotes, storage, packing"),
    ("dog groomer", "appointments, breeds, prices, nervous dogs"),
    ("online craft shop", "orders, delivery times, custom pieces, damaged items"),
    ("web design agency", "new websites, redesigns, hosting, prices"),
    ("charity", "volunteering, donations, events, help with services"),
]

LOCALES = [
    ("UK", "British English spelling, UK phone numbers and towns"),
    ("US", "American English spelling, US phone numbers and cities"),
    ("Canada", "Canadian English, Canadian phone numbers and cities"),
    ("Australia", "Australian English, Australian phone numbers and cities"),
    ("Ireland", "Irish English, Irish phone numbers and towns"),
    ("India", "Indian English as commonly written, Indian phone numbers and cities"),
    ("Pakistan", "Pakistani English as commonly written, Pakistani phone numbers and cities"),
    ("Nigeria", "Nigerian English as commonly written, Nigerian phone numbers and cities"),
]


@dataclass(frozen=True)
class Style:
    key: str
    description: str
    words: tuple[int, int]
    weight: float


STYLES = [
    Style(
        "formal_email",
        "written like a proper email: greeting, two to four short paragraphs, sign-off with a "
        "first name",
        (60, 220),
        1.4,
    ),
    Style(
        "casual_chat",
        "casual and chatty like a text message: lowercase, little punctuation, maybe an emoji or "
        "'thx', 'pls', 'u'",
        (8, 60),
        1.4,
    ),
    Style("short_note", "one or two short sentences, no greeting", (4, 25), 1.2),
    Style(
        "rambling",
        "long and rambling, with background detail and a few digressions before the point",
        (150, 400),
        0.8,
    ),
    Style(
        "non_native",
        "written by someone whose first language is not English: small grammar mistakes, unusual "
        "phrasing, but clear meaning",
        (20, 120),
        1.0,
    ),
    Style(
        "typo_heavy", "typed quickly on a phone: several typos and missing capitals", (10, 80), 0.8
    ),
    Style(
        "angry",
        "frustrated or angry: some capital letters, exclamation marks, blunt",
        (20, 150),
        0.6,
    ),
    Style("polite_brief", "polite and to the point, three or four sentences", (25, 70), 1.0),
]
STYLE_BY_KEY = {s.key: s for s in STYLES}


@dataclass(frozen=True)
class Intent:
    key: str
    label: str  # "ham" | "spam"
    subtype: str  # labelling-guidelines.md subtypes
    channel: str  # "contact_form" | "comment"
    description: str
    weight: float
    styles: tuple[str, ...] = tuple(STYLE_BY_KEY)


_SPAMMY_STYLES = ("formal_email", "casual_chat", "short_note", "non_native", "polite_brief")

INTENTS = [
    # --- genuine contact-form messages ---
    Intent(
        "quote_request",
        "ham",
        "enquiry",
        "contact_form",
        "asks for a price or quote for something the business does, with some detail of the job",
        1.6,
    ),
    Intent(
        "booking",
        "ham",
        "enquiry",
        "contact_form",
        "wants to book, reschedule or check availability for an appointment, table, room or date",
        1.4,
    ),
    Intent(
        "service_question",
        "ham",
        "enquiry",
        "contact_form",
        "asks a question about the service: whether they cover an area, opening hours, what is "
        "included, how long it takes",
        1.3,
    ),
    Intent(
        "customer_problem",
        "ham",
        "support",
        "contact_form",
        "an existing customer with a problem: late order, wrong item, a fault after a job, a "
        "billing mistake",
        1.1,
    ),
    Intent(
        "complaint",
        "ham",
        "support",
        "contact_form",
        "a genuine complaint from an unhappy customer who wants it put right",
        0.6,
    ),
    Intent(
        "follow_up",
        "ham",
        "enquiry",
        "contact_form",
        "follows up on an earlier call, visit or quote and asks what happens next",
        0.8,
    ),
    Intent(
        "vague_callback",
        "ham",
        "enquiry",
        "contact_form",
        "a vague but genuine message asking for someone to call or email back, with little detail",
        0.7,
        ("short_note", "casual_chat", "polite_brief", "typo_heavy", "non_native"),
    ),
    Intent(
        "job_application",
        "ham",
        "other_ham",
        "contact_form",
        "someone applying for a job or work experience at the business itself",
        0.4,
    ),
    Intent(
        "thanks_feedback",
        "ham",
        "other_ham",
        "contact_form",
        "a thank-you or positive feedback after using the business",
        0.4,
    ),
    # --- genuine blog or page comments ---
    Intent(
        "genuine_comment",
        "ham",
        "other_ham",
        "comment",
        "a real reader commenting on a blog post the business published: an opinion, a related "
        "experience or a question to the author",
        0.7,
        ("casual_chat", "short_note", "polite_brief", "non_native", "typo_heavy", "rambling"),
    ),
    # --- solicitation: sales pitches to the business ---
    Intent(
        "seo_pitch",
        "spam",
        "solicitation",
        "contact_form",
        "an unsolicited SEO pitch: claims the site is not ranking on Google's first page and "
        "offers to fix it, often with a fake audit or a 'quick question'",
        1.2,
        _SPAMMY_STYLES,
    ),
    Intent(
        "web_design_pitch",
        "spam",
        "solicitation",
        "contact_form",
        "an unsolicited offer to redesign the website, build an app or fix 'errors' found on the "
        "site, often from an overseas agency",
        1.0,
        _SPAMMY_STYLES,
    ),
    Intent(
        "lead_gen_pitch",
        "spam",
        "solicitation",
        "contact_form",
        "sells leads, email lists, appointment setting, social media management or paid ads to "
        "the business",
        0.9,
        _SPAMMY_STYLES,
    ),
    Intent(
        "guest_post_request",
        "spam",
        "solicitation",
        "contact_form",
        "asks to publish a 'guest post' or buy a link on the site, or offers backlinks",
        0.6,
        _SPAMMY_STYLES,
    ),
    Intent(
        "fake_enquiry_pitch",
        "spam",
        "solicitation",
        "contact_form",
        "pretends to be a customer enquiry in the first line, then turns into a sales pitch for a "
        "marketing or software service",
        0.7,
        _SPAMMY_STYLES,
    ),
    # --- bot and scam spam ---
    Intent(
        "phishing",
        "spam",
        "phishing",
        "contact_form",
        "a scam: the domain or website is about to expire, an invoice or legal/copyright notice "
        "is attached, or the account needs verifying via a link",
        0.8,
        ("formal_email", "short_note", "polite_brief", "non_native"),
    ),
    Intent(
        "advance_fee_order",
        "spam",
        "phishing",
        "contact_form",
        "a fraudulent large order: wants to buy in bulk, pay by cheque or with an overpayment, or "
        "ship abroad through 'their own shipper'",
        0.4,
        ("formal_email", "non_native", "polite_brief"),
    ),
    Intent(
        "crypto_investment",
        "spam",
        "crypto",
        "contact_form",
        "promotes a crypto, forex or investment scheme with guaranteed returns",
        0.5,
        ("casual_chat", "short_note", "non_native", "formal_email"),
    ),
    Intent(
        "loan_offer",
        "spam",
        "bot",
        "contact_form",
        "a lender or broker cold-pitching business loans, funding or merchant cash advances with "
        "'instant approval' to the business owner (the writer is offering money, not asking "
        "for it)",
        0.4,
        _SPAMMY_STYLES,
    ),
    Intent(
        "comment_link_spam",
        "spam",
        "bot",
        "comment",
        "a short generic comment ('great post, very informative') that exists only to leave a "
        "link to an unrelated site",
        1.0,
        ("short_note", "casual_chat", "non_native", "polite_brief"),
    ),
    Intent(
        "keyword_stuffing",
        "spam",
        "bot",
        "contact_form",
        "machine-written junk: repeated keywords, lists of links, or text that makes no sense",
        0.4,
        ("short_note", "non_native"),
    ),
]
INTENT_BY_KEY = {i.key: i for i in INTENTS}

SYSTEM_PROMPT = (
    "You write realistic example messages for testing a spam filter for small-business websites. "
    "Write exactly what a real person (or a real spammer) would type. Never explain, never add "
    "notes, never say it is an example. Reply with JSON only."
)


@dataclass(frozen=True)
class Request:
    request_id: str
    template_id: str
    intent: str
    label: str
    subtype: str
    channel: str
    style: str
    business: str
    locale: str
    seed: int
    messages: tuple[tuple[str, str], ...]  # (role, content)

    def chat(self) -> list[dict[str, str]]:
        return [{"role": role, "content": content} for role, content in self.messages]

    def meta(self) -> dict[str, object]:
        row = asdict(self)
        del row["messages"]
        return row


def template_id(intent: Intent, style: Style) -> str:
    return f"p{PROMPT_VERSION}:{intent.channel}:{intent.key}:{style.key}"


def user_prompt(
    intent: Intent, style: Style, business: tuple[str, str], locale: tuple[str, str]
) -> str:
    if intent.label == "spam":
        who = f"a spammer or scammer writing TO the {business[0]} (not one of its customers)"
        extras = (
            "Some messages include a link, an email address or a phone number; others ask for a "
            "reply instead."
        )
    else:
        who = f"a real customer or reader of the {business[0]}"
        # Without this, genuine messages never contain links and the model learns "link = spam".
        extras = (
            "About one in four writers mentions a link (their own website, a photo album, a "
            "listing or the product page they mean) or adds their email address; the rest don't."
        )
    where = (
        "the contact form on the website"
        if intent.channel == "contact_form"
        else "the comments under a blog post on the website"
    )
    low, high = style.words
    return (
        f"Business: a {business[0]} ({business[1]}).\n"
        f"Where the message is posted: {where}.\n"
        f"Who writes it: {who}. What it is: {intent.description}.\n"
        f"{extras}\n"
        f"Writing style: {style.description}.\n"
        f"Length: about {low}-{high} words each.\n"
        f"Locale: {locale[0]}: {locale[1]}.\n\n"
        f"Write {MESSAGES_PER_REQUEST} different messages, each from a different writer with "
        "different details. Make them varied and believable; a careful human must not be able to "
        "tell they are generated. Invent names. Use made-up company names; any website or email "
        "must use a made-up domain (never a real brand). Do not use placeholders like [Name].\n\n"
        'Reply with a JSON array of strings only, like ["first message", "second message"].'
    )


def build_requests(n: int, seed: int, generator: str) -> list[Request]:
    """Sample ``n`` generation requests. Deterministic for a given (seed, generator)."""
    rng = random.Random(f"{seed}:{generator}:{PROMPT_VERSION}")
    intent_weights = [i.weight for i in INTENTS]
    out: list[Request] = []
    for k in range(n):
        intent = rng.choices(INTENTS, weights=intent_weights)[0]
        styles = [STYLE_BY_KEY[key] for key in intent.styles]
        style = rng.choices(styles, weights=[s.weight for s in styles])[0]
        business = rng.choice(BUSINESSES)
        locale = rng.choice(LOCALES)
        rid = hashlib.sha256(f"{generator}:{seed}:{k}".encode()).hexdigest()[:16]
        out.append(
            Request(
                request_id=rid,
                template_id=template_id(intent, style),
                intent=intent.key,
                label=intent.label,
                subtype=intent.subtype,
                channel=intent.channel,
                style=style.key,
                business=business[0],
                locale=locale[0],
                seed=seed,
                messages=(
                    ("system", SYSTEM_PROMPT),
                    ("user", user_prompt(intent, style, business, locale)),
                ),
            )
        )
    return out


_ARRAY = re.compile(r"\[.*\]", re.DOTALL)
_REFUSALS = (
    "as an ai",
    "i cannot assist",
    "i can't assist",
    "i can't help with",
    "i cannot help with",
    "example message",
    "[name]",
    "[your name]",
    "lorem ipsum",
    "language model",
)
# The model's own scaffolding at the start of a message ("1.", "Message 2:", "Subject:"); a numbered
# list inside a genuine email is fine, so only the start is checked.
_LEADING_LABEL = re.compile(
    r"^\s*((message|email|comment|msg)\s*#?\d+\s*[:.)-]|\d+[.)]\s|subject\s*:)", re.I
)
_PLACEHOLDER = re.compile(r"\[[A-Z][A-Za-z ]{1,25}\]|\{\{?[A-Za-z_ ]{2,25}\}?\}")


def parse_response(text: str) -> list[str]:
    """Pull the JSON array of messages out of a model reply; [] if there isn't one."""
    match = _ARRAY.search(text)
    if not match:
        return []
    try:
        items = json.loads(match.group(0))
    except json.JSONDecodeError:
        return []
    if not isinstance(items, list):
        return []
    out = []
    for item in items:
        if isinstance(item, dict):
            item = item.get("text") or item.get("message") or ""
        if isinstance(item, str):
            out.append(item)
    return out


def clean_messages(texts: list[str]) -> list[str]:
    """Drop empty, refusal-like, placeholder, scaffolded and duplicate messages."""
    seen: set[str] = set()
    out = []
    for text in texts:
        text = text.strip()
        key = " ".join(text.lower().split())
        if len(key) < 3 or key in seen or any(marker in key for marker in _REFUSALS):
            continue
        if _LEADING_LABEL.match(text) or _PLACEHOLDER.search(text):
            continue
        seen.add(key)
        out.append(text)
    return out


In [ ]:
%%writefile gen.py
"""Run one generator model over the request grid (runs on a GPU: Kaggle or Colab).

    python gen.py --generator qwen2.5-7b --model Qwen/Qwen2.5-7B-Instruct-AWQ --quantization awq \
        --fallback-model Qwen/Qwen2.5-7B-Instruct --fallback-4bit --n-requests 200 --out out.jsonl

Rows are appended and flushed after every batch, and requests already in the output are skipped,
so a session that times out can simply be run again. vLLM is used when it works; otherwise plain
transformers (slower). Standard library only at import time, so the notebook can embed it.
"""

from __future__ import annotations

import argparse
import json
import sys
import time
from collections.abc import Iterator, Sequence
from pathlib import Path
from typing import Any, TextIO

try:
    from laas.synth.spec import (
        PROMPT_VERSION,
        Request,
        build_requests,
        clean_messages,
        parse_response,
    )
except ImportError:  # inside the notebook, spec.py sits next to this file
    from spec import (  # type: ignore[no-redef,import-not-found]
        PROMPT_VERSION,
        Request,
        build_requests,
        clean_messages,
        parse_response,
    )


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


def batches(items: Sequence[Request], size: int) -> Iterator[Sequence[Request]]:
    for start in range(0, len(items), size):
        yield items[start : start + size]


def done_ids(path: Path) -> set[str]:
    if not path.exists():
        return set()
    with path.open(encoding="utf-8") as fh:
        return {json.loads(line)["request_id"] for line in fh if line.strip()}


def write(fh: TextIO, req: Request, reply: str, args: argparse.Namespace, checkpoint: str) -> int:
    texts = clean_messages(parse_response(reply))
    for i, text in enumerate(texts):
        row = {
            **req.meta(),
            "generator": args.generator,
            "checkpoint": checkpoint,
            "prompt_version": PROMPT_VERSION,
            "index": i,
            "text": text,
        }
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")
    if not texts:  # record the attempt so a resume doesn't retry a request that keeps failing
        fh.write(json.dumps({**req.meta(), "generator": args.generator, "failed": True}) + "\n")
    return len(texts)


def run_vllm(reqs: Sequence[Request], args: argparse.Namespace, fh: TextIO) -> int:
    log("STAGE importing vLLM")
    from vllm import LLM, SamplingParams

    log(f"STAGE loading {args.model} with vLLM (the first run downloads it: a few minutes)")
    started = time.time()
    llm = LLM(
        model=args.model,
        dtype="half",  # T4 GPUs have no bfloat16
        quantization=args.quantization or None,
        max_model_len=args.max_model_len,
        gpu_memory_utilization=0.90,
        seed=args.seed,
    )
    log(f"STAGE model loaded in {time.time() - started:.0f}s; generating")
    tokenizer = llm.get_tokenizer()
    params = SamplingParams(temperature=args.temperature, top_p=0.95, max_tokens=args.max_tokens)
    total = 0
    n_batches = -(-len(reqs) // args.batch)
    for i, chunk in enumerate(batches(reqs, args.batch)):
        batch_started = time.time()
        prompts = [
            tokenizer.apply_chat_template(r.chat(), tokenize=False, add_generation_prompt=True)
            for r in chunk
        ]
        outputs = llm.generate(prompts, params, use_tqdm=False)
        for req, out in zip(chunk, outputs, strict=True):
            total += write(fh, req, out.outputs[0].text, args, args.model)
        fh.flush()
        log(
            f"PROGRESS batch {i + 1}/{n_batches} ({len(chunk)} requests in "
            f"{time.time() - batch_started:.0f}s): {total:,} messages so far"
        )
    return total


def run_transformers(reqs: Sequence[Request], args: argparse.Namespace, fh: TextIO) -> int:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    checkpoint = args.fallback_model or args.model
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    kwargs: dict[str, Any] = {"torch_dtype": torch.float16, "device_map": "cuda"}
    if args.fallback_4bit:
        from transformers import BitsAndBytesConfig

        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16
        )
    log(f"STAGE loading {checkpoint} with transformers (4-bit: {args.fallback_4bit})")
    started = time.time()
    model = AutoModelForCausalLM.from_pretrained(checkpoint, **kwargs)
    log(f"STAGE model loaded in {time.time() - started:.0f}s; generating")
    total = 0
    n_batches = -(-len(reqs) // args.hf_batch)
    for i, chunk in enumerate(batches(reqs, args.hf_batch)):
        batch_started = time.time()
        prompts = [
            tokenizer.apply_chat_template(r.chat(), tokenize=False, add_generation_prompt=True)
            for r in chunk
        ]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            generated = model.generate(
                **enc,
                do_sample=True,
                temperature=args.temperature,
                top_p=0.95,
                max_new_tokens=args.max_tokens,
                pad_token_id=tokenizer.pad_token_id,
            )
        replies = tokenizer.batch_decode(
            generated[:, enc["input_ids"].shape[1] :], skip_special_tokens=True
        )
        for req, reply in zip(chunk, replies, strict=True):
            total += write(fh, req, reply, args, checkpoint)
        fh.flush()
        log(
            f"PROGRESS batch {i + 1}/{n_batches} ({len(chunk)} requests in "
            f"{time.time() - batch_started:.0f}s): {total:,} messages so far"
        )
    return total


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--generator", required=True, help="Short name, e.g. qwen2.5-7b")
    parser.add_argument("--model", required=True, help="Checkpoint for vLLM")
    parser.add_argument("--quantization", default="", help="e.g. awq (vLLM only)")
    parser.add_argument("--fallback-model", default="", help="Checkpoint for transformers")
    parser.add_argument("--fallback-4bit", action="store_true", help="Load fallback in 4-bit")
    parser.add_argument("--backend", choices=["auto", "vllm", "transformers"], default="auto")
    parser.add_argument("--n-requests", type=int, default=200)
    parser.add_argument("--seed", type=int, default=20261018)
    parser.add_argument("--out", type=Path, required=True)
    # Small enough that progress shows every minute or two; vLLM still batches efficiently.
    parser.add_argument("--batch", type=int, default=32)
    parser.add_argument("--hf-batch", type=int, default=16)
    parser.add_argument("--max-tokens", type=int, default=1600)
    parser.add_argument("--max-model-len", type=int, default=4096)
    parser.add_argument("--temperature", type=float, default=0.95)
    args = parser.parse_args(argv)

    reqs = build_requests(args.n_requests, args.seed, args.generator)
    done = done_ids(args.out)
    todo = [r for r in reqs if r.request_id not in done]
    log(
        f"{args.generator}: {len(reqs):,} requests, {len(done):,} already done, {len(todo):,} to go"
    )
    if not todo:
        return 0

    args.out.parent.mkdir(parents=True, exist_ok=True)
    with args.out.open("a", encoding="utf-8") as fh:
        if args.backend in ("auto", "vllm"):
            try:
                total = run_vllm(todo, args, fh)
                log(f"Done with vLLM: {total:,} messages")
                return 0
            except Exception as exc:  # any vLLM failure (install, CUDA, OOM) falls back
                if args.backend == "vllm":
                    raise
                log(f"vLLM failed ({type(exc).__name__}: {exc}); falling back to transformers")
                fh.flush()
                todo = [r for r in todo if r.request_id not in done_ids(args.out)]
        total = run_transformers(todo, args, fh)
        log(f"Done with transformers: {total:,} messages")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
print("▶ Step 3/5: checking the generation code")
import importlib, spec, gen
importlib.reload(spec); importlib.reload(gen)
print(f"   {len(spec.INTENTS)} intents · {len(spec.STYLES)} writing styles · "
      f"{len(spec.BUSINESSES)} business types · {len(spec.LOCALES)} locales")
print("✅ Step 3 done")


## 4. Generate (resumable: re-run this cell if a session times out)

In [ ]:
print("▶ Step 4/5: generating messages (pilot: about 20-40 minutes with transformers).")
print("   A status line prints every 30 seconds. Re-run this cell if a session times out: it resumes.")
import subprocess, sys, time, torch

def command(g, python, backend):
    cmd = [python, "-u", "gen.py", "--generator", g["generator"], "--model", g["model"],
           "--backend", backend,
           "--n-requests", str(N_REQUESTS), "--out", f"{OUT_DIR}/{g['generator']}.jsonl"]
    if g["quantization"]:
        cmd += ["--quantization", g["quantization"]]
    if g["fallback_model"]:
        cmd += ["--fallback-model", g["fallback_model"]]
    if g["fallback_4bit"]:
        cmd.append("--fallback-4bit")
    return cmd

def messages_written(g):
    path = f"{OUT_DIR}/{g['generator']}.jsonl"
    if not os.path.exists(path):
        return 0
    with open(path, encoding="utf-8") as fh:
        return sum(1 for line in fh if line.strip() and '"failed": true' not in line)

def latest_status(g):
    try:
        with open(f"{OUT_DIR}/{g['generator']}.log", encoding="utf-8", errors="replace") as fh:
            lines = [l.strip() for l in fh if l.strip()]
    except FileNotFoundError:
        return "starting..."
    ours = [l for l in lines if any(k in l for k in ("STAGE", "PROGRESS", "Done", "failed", "Error"))]
    return (ours or lines or ["starting..."])[-1][:150]

def watch(running, started):
    while any(proc.poll() is None for _, proc in running):
        time.sleep(30)
        minutes, seconds = divmod(int(time.time() - started), 60)
        for g, proc in running:
            state = "running" if proc.poll() is None else f"finished (exit {proc.returncode})"
            print(f"   [{minutes:02d}:{seconds:02d}] {g['generator']:<11} {messages_written(g):>6,} msgs · "
                  f"{state} · {latest_status(g)}", flush=True)

def launch(g, gpu_index, python, backend):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu_index), "PYTHONUNBUFFERED": "1"}
    log = open(f"{OUT_DIR}/{g['generator']}.log", "a")
    print(f"   started {g['generator']} on GPU {gpu_index} with {backend}", flush=True)
    proc = subprocess.Popen(command(g, python, backend), env=env, stdout=log, stderr=subprocess.STDOUT)
    return g, proc

def run_phase(models, python, backend):
    # Run models (in parallel when there are 2+ GPUs); return the ones that failed.
    failed = []
    if gpus >= 2:
        running = [launch(g, i, python, backend) for i, g in enumerate(models)]
        watch(running, started)
    else:
        running = []
        for g in models:
            one = [launch(g, 0, python, backend)]
            watch(one, started)
            running += one
    for g, proc in running:
        if proc.returncode != 0:
            print(f"   {g['generator']} stopped with exit code {proc.returncode} ({backend})")
            failed.append(g)
    return failed

gpus = torch.cuda.device_count()
print(f"   {gpus} GPU(s): running {'both models in parallel' if gpus >= 2 else 'the models one after the other'}")
started = time.time()
todo = list(GENERATORS)
if VLLM_PYTHON:
    todo = run_phase(todo, VLLM_PYTHON, "vllm")
    if todo:
        print("   switching those to transformers; finished requests are kept and skipped")
if todo:
    todo = run_phase(todo, sys.executable, "transformers")

print(f"   all models finished in {(time.time() - started) / 60:.0f} min")
ok = True
for g in GENERATORS:
    count = messages_written(g)
    if g not in todo and count > 0:
        print(f"   ✓ {g['generator']}: {count:,} messages · {latest_status(g)}")
    else:
        ok = False
        print(f"   ✗ {g['generator']}: {count:,} messages. Last log lines:")
        with open(f"{OUT_DIR}/{g['generator']}.log", encoding="utf-8", errors="replace") as fh:
            print("".join(fh.readlines()[-20:]))
print("✅ Step 4 done" if ok else "❌ Step 4 had a problem: copy the log lines above and send them over")


## 5. Check the output and package it

In [ ]:
print("▶ Step 5/5: checking the output and packaging it")
import collections, glob, json, random, shutil

rows = []
for path in glob.glob(f"{OUT_DIR}/*.jsonl"):
    rows += [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
ok = [r for r in rows if not r.get("failed")]
failed = len(rows) - len(ok)
print(f"   {len(ok):,} messages, {failed:,} failed requests")
print("   by generator:", dict(collections.Counter(r["generator"] for r in ok)))
print("   by label:", dict(collections.Counter(r["label"] for r in ok)))
print("   by style:", dict(collections.Counter(r["style"] for r in ok)))
print("   by intent:", collections.Counter(r["intent"] for r in ok).most_common())

random.seed(0)
for label in ("ham", "spam"):
    pool = [r for r in ok if r["label"] == label]
    print(f"\n===== random {label} samples (read these: do they look real?) =====")
    for r in random.sample(pool, k=min(6, len(pool))):
        tag = f"{r['generator']} | {r['intent']} | {r['style']} | {r['business']}"
        print(f"\n[{tag}]\n{r['text'][:600]}")

archive = shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
size_mb = os.path.getsize(archive) / 1e6
print(f"\n   packaged: {archive} ({size_mb:.1f} MB)")

# Download: automatic on Colab; a clickable link on Kaggle (Kaggle doesn't allow notebooks to start
# downloads, and a background "Save & Run All" run has no browser, so use the Output tab there).
try:
    from google.colab import files
    print("   starting the download in your browser (Colab)...")
    files.download(archive)
except ImportError:
    from IPython.display import FileLink, display
    print("   click to download (interactive Kaggle session), or use the Output tab:")
    display(FileLink(os.path.relpath(archive), result_html_prefix="   ⬇ "))
print("✅ Step 5 done: unzip synthetic.zip into training/data/synthetic/ in the repo")
